# Test Treatment Condition API

This notebook tests the new queryable treatment condition data classes:
- `TreatmentCondition` with `Drug` and `CellLine` objects
- Lazy loading of pathway activities
- Factory methods on `L2LData`

In [2]:
from l2l_bench import L2LData, TreatmentCondition, Drug, CellLine, DriverMutation

In [3]:
# initialize L2LData (loads metadata)
data = L2LData()

HuggingFace authentication configured
Loading metadata from tahoebio/Tahoe-100M...


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Gene metadata: 62710 genes


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Drug metadata: 379 drugs


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Cell line metadata: 102 unique cell lines


Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

  Sample metadata: 1344 samples
Metadata loaded successfully.


## Test Drug API

In [4]:
# get a drug object
trametinib = data.get_drug("Trametinib")
print(f"Drug: {trametinib.name}")
print(f"MOA (fine): {trametinib.moa_fine}")
print(f"MOA (broad): {trametinib.moa_broad}")
print(f"Targets: {trametinib.targets}")
print(f"Human approved: {trametinib.human_approved}")
print(f"Clinical trials: {trametinib.clinical_trials}")
print(f"PubChem CID: {trametinib.pubchem_cid}")
print(f"SMILES: {trametinib.canonical_smiles[:50]}..." if trametinib.canonical_smiles else "No SMILES")

Drug: Trametinib
MOA (fine): MEK inhibitor
MOA (broad): inhibitor/antagonist
Targets: ['MAP2K1', 'MAP2K2']
Human approved: True
Clinical trials: True
PubChem CID: 11707110
SMILES: CC1=C2C(=C(N(C1=O)C)NC3=C(C=C(C=C3)I)F)C(=O)N(C(=O...


## Test CellLine API

In [5]:
# get a cell line object
a549 = data.get_cell_line("A549")
print(f"Cell line: {a549.name}")
print(f"Organ: {a549.organ}")
print(f"Number of driver mutations: {len(a549.driver_mutations)}")
print(f"\nDriver mutations:")
for mut in a549.driver_mutations:
    print(f"  - {mut.gene_symbol} {mut.protein_effect} ({mut.var_type}, {mut.mechanism}, {mut.gene_type})")

Cell line: A549
Organ: Lung
Number of driver mutations: 6

Driver mutations:
  - CDKN2A DEL (Deletion, LoF, Suppressor)
  - CDKN2B DEL (Deletion, LoF, Suppressor)
  - KRAS p.G12S (Missense, GoF, Oncogene)
  - SMARCA4 p.Q729fs (Frameshift, LoF, Suppressor)
  - STK11 p.Q37* (Stopgain, LoF, Suppressor)
  - ZFHX3 p.L1473F (Missense, None, Suppressor)


In [6]:
# test mutation query methods
print(f"Has KRAS mutation: {a549.has_mutation('KRAS')}")
print(f"Has EGFR mutation: {a549.has_mutation('EGFR')}")
print(f"\nOncogenes: {[m.gene_symbol for m in a549.get_oncogenes()]}")
print(f"Tumor suppressors: {[m.gene_symbol for m in a549.get_tumor_suppressors()]}")

Has KRAS mutation: True
Has EGFR mutation: False

Oncogenes: ['KRAS']
Tumor suppressors: ['CDKN2A', 'CDKN2B', 'SMARCA4', 'STK11', 'ZFHX3']


## Test TreatmentCondition with Pathway Activities

In [33]:
# get a treatment condition with pathway activities
treatment = data.get_treatment("Dabrafenib", 5.0, "A549")
print(f"Treatment: {treatment}")
print(f"\nDrug: {treatment.drug.name} ({treatment.drug.moa_fine})")
print(f"Cell line: {treatment.cell_line.name} ({treatment.cell_line.organ})")
print(f"Concentration: {treatment.concentration} {treatment.concentration_unit}")

Treatment: Dabrafenib_5.0uM_A549

Drug: Dabrafenib (RAF inhibitor)
Cell line: A549 (Lung)
Concentration: 5.0 uM


In [32]:
treatment.pathway_activities.loc[treatment.pathway_activities['pathway'].str.contains('MAPK')]

,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line


In [8]:
# test lazy loading of pathway activities
print("Loading pathway activities (lazy load on first access)...")
pathways = treatment.pathway_activities
print(f"Loaded {len(pathways)} pathways")
pathways.head()

Loading pathway activities (lazy load on first access)...
Loaded 1065 pathways


,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
0,rRNA Processing In Nucleus And Cytosol R-HSA-8...,2.164058,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
1,Cellular Response To Starvation R-HSA-9711097,2.158173,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
2,Major Pathway Of rRNA Processing In Nucleolus ...,2.157167,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
3,rRNA Processing R-HSA-72312,2.156543,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
4,Cap-dependent Translation Initiation R-HSA-72737,2.143990,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549


In [9]:
# test significant pathways
significant = treatment.get_significant_pathways(fdr_threshold=0.05)
print(f"Significant pathways (FDR < 0.05): {len(significant)}")
significant.head(10)

Significant pathways (FDR < 0.05): 69


,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
0,rRNA Processing In Nucleus And Cytosol R-HSA-8...,2.164058,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
1,Cellular Response To Starvation R-HSA-9711097,2.158173,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
2,Major Pathway Of rRNA Processing In Nucleolus ...,2.157167,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
3,rRNA Processing R-HSA-72312,2.156543,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
4,Cap-dependent Translation Initiation R-HSA-72737,2.143990,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
5,L13a-mediated Translational Silencing Of Cerul...,2.137359,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
6,Nonsense Mediated Decay (NMD) Enhanced By Exon...,2.129568,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
7,GTP Hydrolysis And Joining Of 60S Ribosomal Su...,2.129363,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
8,Eukaryotic Translation Elongation R-HSA-156842,2.127560,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
9,Formation Of A Pool Of Free 40S Subunits R-HSA...,2.125028,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549


In [10]:
# test top activated pathways
print("Top 10 activated pathways:")
treatment.get_top_activated(n=10)

Top 10 activated pathways:


,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
0,rRNA Processing In Nucleus And Cytosol R-HSA-8...,2.164058,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
1,Cellular Response To Starvation R-HSA-9711097,2.158173,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
2,Major Pathway Of rRNA Processing In Nucleolus ...,2.157167,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
3,rRNA Processing R-HSA-72312,2.156543,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
4,Cap-dependent Translation Initiation R-HSA-72737,2.143990,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
5,L13a-mediated Translational Silencing Of Cerul...,2.137359,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
6,Nonsense Mediated Decay (NMD) Enhanced By Exon...,2.129568,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
7,GTP Hydrolysis And Joining Of 60S Ribosomal Su...,2.129363,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
8,Eukaryotic Translation Elongation R-HSA-156842,2.127560,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549
9,Formation Of A Pool Of Free 40S Subunits R-HSA...,2.125028,0.0,0.0,RPS2;RPL13A;RPS12;RPL11;RPS8;RPS27A;RPS27;RPL3...,Trametinib,0.5,A549


In [11]:
# test top repressed pathways
print("Top 10 repressed pathways:")
treatment.get_top_repressed(n=10)

Top 10 repressed pathways:


,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
47,SHC1 Events In ERBB2 Signaling R-HSA-1250196,-1.672579,0.002079,0.533183,PTPN12;PRKCA;EGFR;NRG1,Trametinib,0.5,A549
51,Non-integrin membrane-ECM Interactions R-HSA-3...,-1.657274,0.004246,0.382001,LAMC2;PRKCA;ITGB5;ITGA2;ITGA6;ITGB4;LAMA5;PDGF...,Trametinib,0.5,A549
54,ER To Golgi Anterograde Transport R-HSA-199977,-1.649875,0.000000,0.297821,CD55;COPA;SEC31A;SEC24D;SCFD1;COG5;GOLGB1;COG6...,Trametinib,0.5,A549
56,G Alpha (S) Signaling Events R-HSA-418555,-1.627286,0.000000,0.362837,PDE10A;PRKCA;PDE4D;PDE7B;GNB1;GPR39;PDE3A;PDE2...,Trametinib,0.5,A549
58,Opioid Signaling R-HSA-111885,-1.622420,0.002283,0.316673,PRKCA;PDE4B;PDE4D;PDE1C;GNB1;ITPR1;CREB1;PPP3R...,Trametinib,0.5,A549
63,COPII-mediated Vesicle Transport R-HSA-204005,-1.605987,0.004577,0.357159,SEC31A;SEC24D;SCFD1;SEC24A;SEC16A;SEC23IP;LMAN...,Trametinib,0.5,A549
64,DARPP-32 Events R-HSA-180024,-1.604912,0.008439,0.312828,PRKCA;PDE4B;PDE4D,Trametinib,0.5,A549
65,Syndecan Interactions R-HSA-3000170,-1.602535,0.004065,0.285223,PRKCA;ITGB5;ITGA2;ITGA6;ITGB4;TGFB1;ITGB1,Trametinib,0.5,A549
66,Regulation Of RUNX1 Expression And Activity R-...,-1.599526,0.015625,0.267254,AGO2;CCND1;CBFB;RUNX1;AGO3,Trametinib,0.5,A549
67,Laminin Interactions R-HSA-3000157,-1.598723,0.004141,0.243509,LAMC2;ITGA2;ITGA3;ITGA6;ITGB4;LAMA5;LAMB1;ITGB1,Trametinib,0.5,A549


In [12]:
# test get_pathway method
mapk_pathway = treatment.get_pathway("MAPK")
if mapk_pathway is not None:
    print(f"MAPK pathway:")
    print(mapk_pathway)
else:
    print("MAPK pathway not found")

MAPK pathway:
pathway          MAP3K8 (TPL2)-dependent MAPK1/3 Activation R-H...
nes                                                       1.360494
pvalue                                                    0.155894
fdr                                                       0.235324
leading_edge                               RPS27A;MAP3K8;UBB;UBA52
drug                                                    Trametinib
concentration                                                  0.5
cell_line                                                     A549
Name: 311, dtype: object


## Test Multiple Treatments

In [13]:
# test another cell line
lovo_treatment = data.get_treatment("Trametinib", 0.5, "LoVo")
print(f"Treatment: {lovo_treatment}")
print(f"Cell line: {lovo_treatment.cell_line.name} ({lovo_treatment.cell_line.organ})")
print(f"Has KRAS mutation: {lovo_treatment.cell_line.has_mutation('KRAS')}")
print(f"\nDriver mutations:")
for mut in lovo_treatment.cell_line.driver_mutations[:5]:
    print(f"  - {mut.gene_symbol} {mut.protein_effect}")

Treatment: Trametinib_0.5uM_LoVo
Cell line: LoVo (Bowel)
Has KRAS mutation: True

Driver mutations:
  - ACVR2A p.K437fs
  - APC p.M1431fs
  - APC p.R1114*
  - APC p.R2816Q
  - ARID1A p.F1924fs


In [14]:
# load pathway activities for LoVo treatment
lovo_pathways = lovo_treatment.pathway_activities
print(f"Loaded {len(lovo_pathways)} pathways for LoVo")
lovo_treatment.get_significant_pathways().head(10)

Loaded 1099 pathways for LoVo


,pathway,nes,pvalue,fdr,leading_edge,drug,concentration,cell_line
0,SRP-dependent Cotranslational Protein Targetin...,2.368580,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
1,Cap-dependent Translation Initiation R-HSA-72737,2.347427,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
2,Formation Of A Pool Of Free 40S Subunits R-HSA...,2.337124,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
3,GTP Hydrolysis And Joining Of 60S Ribosomal Su...,2.336395,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
4,Nonsense Mediated Decay (NMD) Enhanced By Exon...,2.335668,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
5,Eukaryotic Translation Elongation R-HSA-156842,2.327146,0.0,0.0,RPL26;RPS6;RPL7;RPS14;EEF1G;RPLP0;RPL37;RPS19;...,Trametinib,0.5,LoVo
6,L13a-mediated Translational Silencing Of Cerul...,2.326383,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
7,Eukaryotic Translation Termination R-HSA-72764,2.325032,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
8,Viral mRNA Translation R-HSA-192823,2.311685,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo
9,Nonsense Mediated Decay (NMD) Independent Of E...,2.310950,0.0,0.0,RPL26;RPS6;RPL7;RPS14;RPLP0;RPL37;RPS19;RPL11;...,Trametinib,0.5,LoVo


## Summary

The new queryable treatment condition API provides:

1. **Drug metadata** via `treatment.drug`:
   - MOA (mechanism of action)
   - Targets
   - SMILES structure
   - PubChem ID

2. **Cell line metadata** via `treatment.cell_line`:
   - Organ/tissue of origin
   - Driver mutations with full annotation
   - Query methods: `has_mutation()`, `get_oncogenes()`, `get_tumor_suppressors()`

3. **Pathway activities** via `treatment.pathway_activities`:
   - Lazy loaded on first access
   - Query methods: `get_significant_pathways()`, `get_top_activated()`, `get_top_repressed()`, `get_pathway()`